In [1]:
print("hi")

hi


In [1]:
import json
import re
import os
from PIL import Image
import pdfplumber
import torch
import cv2
import numpy as np
# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)

# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


# =========================
# CROP TABLES
# =========================
def crop_claim_tables(pdf_path, output_dir="Argus_output"):
    """
    Argus EOPs mark the start of a claim table with 'Claim #:' (WITH a space,
    unlike Allied's 'Claim#:') and the end with the 'Column Totals' row.

    Just below 'Column Totals' sits:
        Patient's Responsibility: $xxx.xx
        Other Insurance Credits:  $xxx.xx
        Total Payment:            $xxx.xx
    so we grab extra vertical room below 'Column Totals' to capture those,
    same as the Allied script does.
    """
    os.makedirs(output_dir, exist_ok=True)
    cropped_images = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            print(f"\n📄 Processing Page {page_num}")

            claim_hits = page.search("Claim #:")
            if not claim_hits:
                claim_hits = page.search("Claim#:")  # fallback in case spacing varies

            total_hits = page.search("Column Totals")

            if not claim_hits or not total_hits:
                continue

            table_count = min(len(claim_hits), len(total_hits))

            for idx in range(table_count):
                # Include a bit above the Claim # line so 'Provider:' header
                # info (name/address) is captured too.
                start_y = claim_hits[idx]["top"] - 40
                end_y   = total_hits[idx]["bottom"] + 45  # extra room for
                                                           # Patient's Responsibility /
                                                           # Other Insurance Credits /
                                                           # Total Payment lines below

                if start_y >= end_y:
                    continue

                bbox = (0, max(start_y, 0), page.width, end_y)
                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )
                cropped_page.to_image(resolution=500).save(image_path)
                print(f"✅ Saved: {image_path}")

                expected_rows = count_service_rows(page, start_y, end_y)

                cropped_images.append({
                    "page":          page_num,
                    "table":         idx + 1,
                    "image_path":    image_path,
                    "expected_rows": expected_rows,
                })

    return cropped_images


def count_service_rows(page, region_top, region_bottom):
    """
    Argus rows start with a single 'Service Date' value, e.g. '07/20/2022'
    (NOT a date range like Allied's '09/24-09/24/2024'). We count unique
    y-positions matching that pattern inside the cropped region as a proxy
    for row count.
    """
    words = page.extract_words()
    line_text = {}
    for w in words:
        y = round(float(w["top"]), 1)
        if region_top <= y <= region_bottom:
            line_text.setdefault(y, []).append(w["text"])

    date_row_pattern = re.compile(r"^\d{2}/\d{2}/\d{4}$")
    service_rows = set()
    for y, words_on_line in line_text.items():
        for tok in words_on_line:
            if date_row_pattern.match(tok):
                service_rows.add(y)
                break

    return len(service_rows)


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums  = np.sum(thresh, axis=1)
    th    = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]
    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)
    return img


AMOUNT_FIELDS = {
    "submitted_charges", "allowed_amount", "provider_responsibility",
    "copay_amount", "not_payable", "coins_copay", "previously_paid",
    "payable_amount", "patients_responsibility",
    "other_insurance_credits", "total_payment",
}


def convert_amounts_to_string(obj):
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            if k in AMOUNT_FIELDS:
                try:
                    new_obj[k] = f"{float(str(v).replace('$', '').replace(',', '').strip()):.2f}"
                except Exception:
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)
        return new_obj
    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]
    else:
        return obj


# =========================
# PROMPT — uses pdf_name as eob_id
# =========================
def build_prompt(pdf_name: str) -> str:
    return f"""
Extract structured data from this Argus Dental & Vision Explanation of Payment (EOP) claim table.

STRICT RULES

1. Extract ONLY the following header fields, found above the service table:
   - patient_name      (value after "Patient:")
   - dob               (value after "DOB:")

2. Extract ALL service rows exactly as shown in the table body.

3. Preserve row order.

4. Do NOT skip duplicate rows (e.g. same Service Code on two different teeth
   are two separate rows).

5. Read the table strictly LEFT to RIGHT.

6. Every service object must correspond to ONE visible table row.

7. Never merge two rows.

8. Never create rows that do not exist.

9. Never use the "Column Totals" row values inside service rows.

10. Stop reading service rows when the row labeled "Column Totals" is reached.

11. Extract the "Column Totals" row separately into the "totals" object.

12. Extract "Patient's Responsibility", "Other Insurance Credits", and
    "Total Payment" (found below the totals row) into the "totals" object
    as patients_responsibility, other_insurance_credits, and total_payment.

13. Ignore: the "Service Description" table and "Reason Code Description"
    table at the bottom of the page, and the "Payment Details" / "Claims
    Appeal" sections — do not extract their contents as service rows.

14. Money values: remove "$" and commas, keep decimals, return as strings.
    Example: "$1,234.00" -> "1234.00"

15. If a field is blank in the table return "".

16. Never infer missing values.

17. Return ONLY valid JSON — no markdown, no explanation, no comments.

COLUMN MAPPING (service rows)

service_date            <- Service Date        (e.g. "07/20/2022")
service_code            <- Service Code
submitted_charges       <- Submitted Charges
allowed_amount          <- Allowed Amount
provider_responsibility <- Provider Responsibility
copay_amount            <- Copay Amount
not_payable             <- Not Payable
coins_copay             <- Co-Ins/Co-Pay
previously_paid         <- Previously Paid
payable_amount          <- Payable Amount

OUTPUT JSON SCHEMA
OUTPUT JSON SCHEMA

{{

  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "dob": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [

    {{

      "service_date": {{
        "value": "",
        "confidence": 0.0
      }},

      "service_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "submitted_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "allowed_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "provider_responsibility": {{
        "value": "",
        "confidence": 0.0
      }},

      "copay_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "not_payable": {{
        "value": "",
        "confidence": 0.0
      }},

      "coins_copay": {{
        "value": "",
        "confidence": 0.0
      }},

      "previously_paid": {{
        "value": "",
        "confidence": 0.0
      }},

      "payable_amount": {{
        "value": "",
        "confidence": 0.0
      }}

    }}

  ],

  "totals": {{

    "submitted_charges": {{
      "value": "",
      "confidence": 0.0
    }},

    "allowed_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "provider_responsibility": {{
      "value": "",
      "confidence": 0.0
    }},

    "copay_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "not_payable": {{
      "value": "",
      "confidence": 0.0
    }},

    "coins_copay": {{
      "value": "",
      "confidence": 0.0
    }},

    "previously_paid": {{
      "value": "",
      "confidence": 0.0
    }},

    "payable_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "patients_responsibility": {{
      "value": "",
      "confidence": 0.0
    }},

    "other_insurance_credits": {{
      "value": "",
      "confidence": 0.0
    }},

    "total_payment": {{
      "value": "",
      "confidence": 0.0
    }}

  }}

}}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

VALIDATION RULES
Number of service objects must equal the number of visible service rows.
Duplicate service codes must be extracted as separate rows.
Do not include the "Column Totals" row inside services.
Output ONLY JSON.
"""


# =========================
# AMOUNT HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").replace("%", "").strip())
    except ValueError:
        return 0.0


# Fields summed from service rows to check against the "Column Totals" row.
# NOTE: tooth, surface, and reason_code are not numeric, so both are
# excluded from totals validation.
TOTALS_FIELDS = [
    "submitted_charges", "allowed_amount", "provider_responsibility",
    "copay_amount", "not_payable", "coins_copay", "previously_paid",
    "payable_amount",
]


def compute_totals_from_services(services: list) -> dict:
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in TOTALS_FIELDS
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount field in every service row is blank/zero."""
    for svc in services:
        for f in TOTALS_FIELDS:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# =========================
# VALIDATION
# =========================
def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):
    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    field_names = list(compute_totals_from_services([]).keys())   # ADD
    total_fields = len(field_names) 

    if not services:
        field_errors = [                                            # ADD
                    {"field": f, "computed": 0.0, "extracted": None}
                    for f in field_names
                ]
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    if services_are_empty(services):
        msg = "All service rows are empty — model likely failed to extract data"
        print(f"\n❌ {msg}")
        field_errors = [
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, msg, [{"error": "all_service_rows_empty"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)
    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(totals.get(field, "")), 2)
        diff  = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01
        icon   = "✅" if match else "❌"
        status = "MATCH" if match else "MISMATCH"

        if not match:
            has_error = True
            errors.append({
                "type":       "field_mismatch",
                "field":      field,
                "computed":   computed_value,
                "extracted":  extracted_value,
                "difference": diff,
            })

        line = (
            f"{icon} {field:25s} computed={computed_value:<10} "
            f"| extracted={extracted_value:<10} {status}"
        )
        print(line)
        result_validation += "\n" + line

    # Cross-check: Total Payment should equal sum of payable_amount rows
    # plus Other Insurance Credits.
    total_payment_extracted = round(parse_amount(totals.get("total_payment", "")), 2)
    payable_sum   = computed_totals.get("payable_amount", 0.0)
    other_credits = round(parse_amount(totals.get("other_insurance_credits", "")), 2)
    expected_total_payment = round(payable_sum + other_credits, 2)
    total_match = abs(expected_total_payment - total_payment_extracted) <= 0.01
    icon   = "✅" if total_match else "❌"
    status = "MATCH" if total_match else "MISMATCH"
    if not total_match:
        has_error = True
        errors.append({
            "type":       "field_mismatch",
            "field":      "total_payment",
            "computed":   expected_total_payment,
            "extracted":  total_payment_extracted,
            "difference": round(expected_total_payment - total_payment_extracted, 2),
        })
    line = (
        f"{icon} {'total_payment':25s} computed={expected_total_payment:<10} "
        f"| extracted={total_payment_extracted:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line

    extracted_row_count = len(services)
    match  = expected_row_count == extracted_row_count
    icon   = "✅" if match else "❌"
    status = "MATCH" if match else "MISMATCH"

    if not match:
        has_error = True
        errors.append({
            "type":           "row_count_mismatch",
            "expected_rows":  expected_row_count,
            "extracted_rows": extracted_row_count,
        })

    line = (
        f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} "
        f"| extracted={extracted_row_count:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line
    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors,total_fields
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# DENIAL CHECK
# =========================
def check_claim_denied(pdf_path: str) -> str:
    """
    Same generic keyword check as the Allied/Sunlife scripts. The sample
    Argus EOP does not contain a denial, so no stop-phrase is applied here.
    If Argus denial letters use different wording, add a stop_phrase check
    the same way the Sunlife script does.
    """
    denial_keywords = ["denied", "denial"]

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text = page.extract_text()
            if not full_text:
                continue
            searchable = full_text.lower()
            for keyword in denial_keywords:
                if keyword in searchable:
                    print(f"claim denied keyword found: {keyword}")
                    return "denied"

    return "not denied"


# =========================
# RETRY HELPER
# =========================
MAX_RETRIES = 2

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=2500,
                temperature=0.0,
                do_sample=False,
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        services = parsed.get("services", [])
        row_ok   = (len(services) == expected_rows)
        empty_ok = not services_are_empty(services)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt}")
            return parsed

        print(
            f"  ⚠️  attempt {attempt}: rows={len(services)} (expected {expected_rows}), "
            f"all_empty={not empty_ok}"
        )
        last_parsed = parsed

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    return last_parsed


# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/Argus", company_name = "Argus"):
    # pdf_name is used as eob_id throughout
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)


    base_dir          = os.path.join(output_dir, pdf_name)
    cropped_dir       = os.path.join(base_dir, "cropped_images")
    json_output_path  = os.path.join(base_dir, f"{pdf_name}_output.json")

    from output_utils import SUCCESS_DIR, FAILED_DIR
    already_success = os.path.exists(os.path.join(SUCCESS_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    already_failed  = os.path.exists(os.path.join(FAILED_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    if already_success or already_failed:
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_items  = crop_claim_tables(pdf_path, output_dir=cropped_dir)
    is_denied    = check_claim_denied(pdf_path)
    print(f"claim status: {is_denied}")

    final_prompt = build_prompt(pdf_name)

    patients = []

    for idx, item in enumerate(image_items):
        img_path      = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\nProcessing table {idx+1}/{len(image_items)}")

        image_cv  = make_table(img_path)
        image_pil = Image.fromarray(image_cv).convert("RGB")

        parsed = run_model_with_retry(image_pil, final_prompt, expected_rows)

        if parsed is None:
            print(f"❌ Skipping table {idx+1} — model returned nothing usable")
            continue

        model_confidence = calculate_model_confidence(parsed)   # ADD
        parsed = _unwrap_vlm_output(parsed)                     # ADD

        parsed["eob_id"] = pdf_name
        parsed = convert_amounts_to_string(parsed)
        parsed["_expected_rows"] = expected_rows
        parsed["_model_confidence"] = model_confidence 

        print("Extracted:")
        print(json.dumps(parsed, indent=2))

        date_of_service = ""
        if parsed.get("services"):
            date_of_service = parsed["services"][0].get("service_date", "")

        patient_data = {
            "patient_name":     parsed.get("patient_name", ""),
            "dob":              parsed.get("dob", ""),
            "date_of_service":  date_of_service,
            "services":         parsed.get("services", []),
            "totals":           parsed.get("totals", {}),
            "_expected_rows":   expected_rows,
            "_model_confidence": parsed.get("_model_confidence", 0.0),  
        }
        patients.append(patient_data)

    # Validate every patient/claim block
    for patient in patients:
        is_valid, log, errors, total_fields = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", "UNKNOWN"),
            expected_row_count=patient.get("_expected_rows", 0),
        )
        patient["validation"] = {
            "status": is_valid,
            "errors": errors,
        }
        patient["_total_fields"] = total_fields   

    confidence_results = list(patients)                              # ADD
    confidence_score = calculate_eob_confidence(confidence_results)   # ADD

    for patient in patients:                                          # existing loop, now also pops these:
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)      # ADD
        patient.pop("_model_confidence", None)   # ADD

    final = [
        {
            "eob_id":       pdf_name,
            "file_name":pdf_full_name,
            "claim_status": is_denied,
            "payor": "Argus Dental & Vision",
            "provider": "Duc Tang",
            "confidence_score": confidence_score, 
            "patients":     patients,
        }
    ]

    success_path, failed_path = save_split_output(
        final,
        company_name=company_name,
        pdf_name=pdf_name,
        pdf_path=pdf_path,
        cropped_dir=cropped_dir,
    )

    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 15:18:38.814000 2991584 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 15:18:38.832000 2991584 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Argus file

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Argus_dental/pdfs"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_349127805.pdf

📄 Processing Page 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


✅ Saved: EOB_OUTPUT/Argus/349127805/cropped_images/page_1_table_1.png
claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/2


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "Yayoi Lasane",
  "dob": "03/13/1975",
  "services": [
    {
      "service_date": "01/16/2023",
      "service_code": "D4910",
      "submitted_charges": "163.98",
      "allowed_amount": "163.98",
      "provider_responsibility": "0.00",
      "copay_amount": "0.00",
      "not_payable": "32.80",
      "coins_copay": "32.80",
      "previously_paid": "0.00",
      "payable_amount": "131.18"
    },
    {
      "service_date": "01/16/2023",
      "service_code": "D0120",
      "submitted_charges": "65.63",
      "allowed_amount": "65.63",
      "provider_responsibility": "0.00",
      "copay_amount": "0.00",
      "not_payable": "0.00",
      "coins_copay": "0.00",
      "previously_paid": "0.00",
      "payable_amount": "65.63"
    }
  ],
  "totals": {
    "submitted_charges": "229.61",
    "allowed_amount": "229.61",
    "provider_responsibility": "0.00",
    "copay_amount": "0.00",
    "not_payable": "32.80",
    "coins_copay"

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


✅ Saved: EOB_OUTPUT/Argus/263924337/cropped_images/page_1_table_1.png
claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/2


[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "Yayoi Lasane",
  "dob": "03/13/1975",
  "services": [
    {
      "service_date": "07/20/2022",
      "service_code": "D4342",
      "submitted_charges": "126.00",
      "allowed_amount": "120.00",
      "provider_responsibility": "6.00",
      "copay_amount": "0.00",
      "not_payable": "18.00",
      "coins_copay": "12.00",
      "previously_paid": "0.00",
      "payable_amount": "108.00"
    },
    {
      "service_date": "07/20/2022",
      "service_code": "D4342",
      "submitted_charges": "126.00",
      "allowed_amount": "0.00",
      "provider_responsibility": "6.00",
      "copay_amount": "0.00",
      "not_payable": "126.00",
      "coins_copay": "0.00",
      "previously_paid": "0.00",
      "payable_amount": "0.00"
    }
  ],
  "totals": {
    "submitted_charges": "252.00",
    "allowed_amount": "120.00",
    "provider_responsibility": "12.00",
    "copay_amount": "0.00",
    "not_payable": "144.00",
    "coins_cop

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Argus_dental/pdfs/Pmt_EOP_307292396.pdf")


📄 Processing Page 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


✅ Saved: EOB_OUTPUT/Argus/307292396/cropped_images/page_1_table_1.png
claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/2


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "Yayoi Lasane",
  "dob": "03/13/1975",
  "services": [
    {
      "service_date": "10/20/2022",
      "service_code": "D0330",
      "submitted_charges": "89.00",
      "allowed_amount": "89.00",
      "provider_responsibility": "0.00",
      "copay_amount": "0.00",
      "not_payable": "0.00",
      "coins_copay": "0.00",
      "previously_paid": "0.00",
      "payable_amount": "89.00"
    },
    {
      "service_date": "10/20/2022",
      "service_code": "D4910",
      "submitted_charges": "105.00",
      "allowed_amount": "105.00",
      "provider_responsibility": "0.00",
      "copay_amount": "0.00",
      "not_payable": "21.00",
      "coins_copay": "21.00",
      "previously_paid": "0.00",
      "payable_amount": "84.00"
    }
  ],
  "totals": {
    "submitted_charges": "194.00",
    "allowed_amount": "194.00",
    "provider_responsibility": "0.00",
    "copay_amount": "0.00",
    "not_payable": "21.00",
    "coins_copay":

[{'eob_id': '307292396',
  'file_name': 'Pmt_EOP_307292396.pdf',
  'claim_status': 'not denied',
  'payor': 'Argus Dental & Vision',
  'provider': 'Duc Tang',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'Yayoi Lasane',
    'dob': '03/13/1975',
    'date_of_service': '10/20/2022',
    'services': [{'service_date': '10/20/2022',
      'service_code': 'D0330',
      'submitted_charges': '89.00',
      'allowed_amount': '89.00',
      'provider_responsibility': '0.00',
      'copay_amount': '0.00',
      'not_payable': '0.00',
      'coins_copay': '0.00',
      'previously_paid': '0.00',
      'payable_amount': '89.00'},
     {'service_date': '10/20/2022',
      'service_code': 'D4910',
      'submitted_charges': '105.00',
      'allowed_amount': '105.00',
      'provider_responsibility': '0.00',
      'copay_amount': '0.00',
      'not_payable': '21.00',
      'coins_copay': '21.00',
      'previously_paid': '0.00',
      'payable_amount': '84.00'}],
    'totals': {'submi